In [1]:
import sqlite3

import pandas as pd

# خواندن داده‌های پاکسازی‌شده
sales = pd.read_csv("clean_sales.csv")

# اتصال به دیتابیس SQLite
sql = sqlite3.connect("sales.db")

# انتقال داده‌ها به جدول sales
sales.to_sql("sales", sql, if_exists="replace", index=False)

print("Sales table created successfully.")

Sales table created successfully.


In [2]:
# فیلتر کردن تراکنش‌هایی که مبلغ نهایی آن‌ها بیشتر از 500 است
query1 = """
SELECT *
FROM sales
WHERE amount > 500;
"""

result1 = pd.read_sql_query(query1, sql)
print(result1)

   transaction_id  customer_id  age  gender    city product     category  \
0          1001.0          101   25       1  Tehran  Laptop  Electronics   
1          1002.0          102   31       0   Karaj   Phone  Electronics   
2          1007.0          102   31       0   Karaj   Phone  Electronics   

   quantity  unit_price  discount payment_method           created_at  amount  
0       2.0         500        10           Card  2026-09-01 10:30:00   900.0  
1       1.0         800         5           Cash  2026-09-01 11:15:00   760.0  
2       2.0         800         5           Card  2026-09-04 18:30:00  1520.0  


In [3]:
# محاسبه مجموع مبلغ تمام فروش‌ها
query2 = """
SELECT SUM(amount) AS total_sales
FROM sales;
"""

result2 = pd.read_sql_query(query2, sql)
print(result2)

   total_sales
0       3645.0


In [4]:
# محاسبه مجموع فروش برای هر محصول
# محصولات بر اساس بیشترین مبلغ فروش مرتب می‌شوند
query3 = """
SELECT product, SUM(amount) AS total_sales
FROM sales
GROUP BY product
ORDER BY total_sales DESC;
"""

result3 = pd.read_sql_query(query3, sql)
print(result3)

    product  total_sales
0     Phone       2280.0
1    Laptop        900.0
2   Monitor        255.0
3     Mouse        120.0
4  Keyboard         90.0


In [5]:
# محاسبه تعداد تراکنش‌های هر مشتری
# مشتریان بر اساس تعداد تراکنش‌ها از بیشترین به کمترین مرتب می‌شوند
query4 = """
SELECT customer_id, COUNT(transaction_id) AS transaction_count
FROM sales
GROUP BY customer_id
ORDER BY transaction_count DESC;
"""

result4 = pd.read_sql_query(query4, sql)
print(result4)

   customer_id  transaction_count
0          101                  3
1          102                  2
2          104                  1
3          103                  1


In [6]:
# محاسبه میانگین مبلغ خرید برای هر شهر
# شهرها بر اساس میانگین مبلغ خرید مرتب می‌شوند
query5 = """
SELECT city, AVG(amount) AS average_amount
FROM sales
GROUP BY city
ORDER BY average_amount DESC;
"""

result5 = pd.read_sql_query(query5, sql)
print(result5)

     city  average_amount
0   Karaj         1140.00
1  Tehran          318.75
2  Qazvin           90.00


In [7]:
# قطع اتصال به دیتابیس
sql.close()